# 10.a — Ad hoc test: daily threshold indicators as SVM predictors (ERA5)

**Ad hoc test, nothing of the chain is changed**: it reads ERA5 daily (as 1.1.d), the BIO files and training tables of
1.1.d, the final models of 3.g and the masks of 0.1 / 0.5, and writes only in
`<out_root>/test_daily_indices/<route>/<set>/`.

**Question.** The 10 BIO variables are 30-year means of daily values: they cannot express how often a damaging day
occurs (one very hot day at anthesis can sterilise the pollen; one frost can kill the plants). Do indicators of such
days, computed from the daily data, improve the SVM? If not, the runs need no daily bias adjustment for the crops.

**Indicators** (per crop, cell and harvest year 1982–2014, on the calendar of the ERA5 BIO file: sowing, start of phase
2 = anthesis of variant A, maturity):

| | indicator | window | threshold |
|---|---|---|---|
| HEAT | at least one day with Tmax above the critical temperature of the crop | 30 days centred on anthesis | maize 35 °C, wheat 34 °C, rice 36 °C, soybean 39 °C (Gourdji et al. 2013, *Environ. Res. Lett.* 8, 024041: Tcrit of the reproductive period, 30-day window around flowering); potato 30 °C (provisional: tuber initiation and bulking are inhibited around 28–30 °C, Levy & Veilleux 2007, *Am. J. Potato Res.* 84, 487–506) |
| FROST | at least one day with Tmin below 0 °C | from emergence (sowing + 10 days) to anthesis; **winter wheat**: the 30-day window around anthesis (late frost at heading and flowering), since frost in winter is normal for it (vernalisation; the winter cold is in BIO12) — every cell of `Wheat-winter_rf`, and the autumn-sown cells of `Wheat_rf` (sowing September–January in the north, March–July in the south, the rule of 0.1) | 0 °C |
| DRY | a dry spell (pr < 1 mm/day) of at least 15 days | phase 2 (anthesis → maturity) | 1 mm/day (ERA5 has too many drizzle days below it) |
| GDD | growing degree days, Σ max(min(Tmean, 30 °C) − Tbase, 0) | sowing → maturity | Tbase maize, soybean, rice 10 °C, wheat 0 °C, potato 5 °C (common values, McMaster & Wilhelm 1997, *Agric. For. Meteorol.* 87, 291–300, for the method) |

**Two summaries, both tested** (the GDD is a mean in both): `years` = frequency of the years with the event (the
probability of a bad season: HEAT_Y, FROST_Y, DRY_Y); `days` = mean fraction of the days of the window with the event
(HEAT_D, FROST_D; DRY_D = longest dry spell / length of phase 2).

**Sets compared**, same training cells, labels, folds (seed 42) and cost × gamma grid as 3.g: `base` = the 10 BIO
variables of the final models (read from 3.g, not retrained); `years` = 10 BIO + HEAT_Y, FROST_Y, DRY_Y, GDD;
`days` = 10 BIO + HEAT_D, FROST_D, DRY_D, GDD.

**Compared** (fixed before looking at the results), per crop: CV AUC and TSS (sensitivity + specificity − 1 at the
SVM decision); on the training cells with the cross-fitted probability (fold models, as 4.g) and the ECOCROP TMIN of
4.g: share of the MIRCA2000 area and of the presence cells called suitable, absences called suitable; on all land:
suitable area, suitable land north of 50° N outside the cropland (the cold tail of maize), land removed by TMIN. No
novelty test here (the same for all sets), so the numbers of `base` differ slightly from 4.g. Univariate AUC of each
indicator. **Decision rule**: the indicators are worth the daily bias adjustment of the runs if they raise the CV TSS
of most crops by about 0.01 or more (the radiation gave +0.015 to +0.032) without losing MIRCA area.

Run time: two sets × 7 crops of cross-validation (8 × 5 × 5 fits each), about twice 3.g.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
import hashlib
import numpy as np, pandas as pd, xarray as xr, joblib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from pathlib import Path
from joblib import Parallel, delayed
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, confusion_matrix
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu
CFG = cu.load_config()

# ---- parameters
CLIMATE = "era5"                  # daily climate of the configuration (ERA5 regridded onto the EC-Earth3 grid), as 1.1.d
CHOSEN_VARIANT = "A_60d"
CHOSEN_RAD = "no_rad"             # the final models of 3.g: the base of the comparison
CELL_AREA_THR = 0.0005
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2014")
HEAT_TCRIT = {"Maize": 35.0, "Wheat": 34.0, "Rice": 36.0, "Soybean": 39.0, "Potato": 30.0}   # degC, Gourdji et al. 2013; potato provisional
FLOWER_WINDOW = (-15, 15)         # days around anthesis (start of phase 2): 30 days, Gourdji et al. 2013
EMERGENCE = 10                    # days after sowing: frost counted from emergence to anthesis (spring-sown crops)
WINTER_SOWING = {"north": (244, 31), "south": (60, 212)}   # DOY of autumn sowing (Sep-Jan / Mar-Jul, rule of 0.1):
                                  # winter wheat, frost counted in the flowering window instead
FROST_T = 0.0                     # degC
DRY_PR, DRY_SPELL = 1.0, 15       # mm/day, days
GDD_BASE = {"Maize": 10.0, "Soybean": 10.0, "Rice": 10.0, "Wheat": 0.0, "Potato": 5.0}   # degC
GDD_CAP = 30.0                    # degC, upper cut-off of the mean temperature
SETS = {"years": ["HEAT_Y", "FROST_Y", "DRY_Y", "GDD"], "days": ["HEAT_D", "FROST_D", "DRY_D", "GDD"]}
COST_GRID = [0.5, 1, 2, 4, 8, 16, 32, 64]         # as 3.g
GAMMA_GRID = [0.1, 0.5, 1, 2, 4]
N_FOLDS, SEED = 5, 42
N_JOBS = min(12, os.cpu_count() or 1)

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ROUTE = cu.route(CELL_AREA_THR)
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
MODELS = cu.out_dir(CFG, "3_svm_train", f"{ERA5_NAME}_{ROUTE}", SET)
OUT = cu.out_dir(CFG, "test_daily_indices", ROUTE, SET)          # the only folder written

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
I, J = np.nonzero(LAND)
NC = I.size
LAND_MHA = np.nan_to_num(cu.land_area_ha(CFG)) * LAND / 1e6
ANY = cu.mirca_cropland(CFG).values
LAT2 = np.broadcast_to(GRID["lat"][:, None], LAND.shape)
N50_OUT = LAND & ~ANY & (LAT2 > 50)
DT = cu.tmin_warming(CFG)
mha = lambda m: float(np.nansum(LAND_MHA * m))
base_of = lambda crop: crop.split("_")[0].split("-")[0]
print(f"{CLIMATE} {ERA5_PERIOD[0]}-{ERA5_PERIOD[1]} | {NC} land cells | base {SET} | written only in {OUT}")

## 1. Daily climate at the land cells (as 1.1.d)

In [ ]:
t0 = time.time()
CLIM = {}
for v in ("tasmax", "tasmin", "pr"):
    da = cu.read_climate(CFG, CLIMATE, v, GRID, ERA5_PERIOD)
    CLIM[v] = cu.land_series(da, (I, J))
    print(f"  {v:6s} {CLIM[v].shape} ({da.attrs['units']}) read in {(time.time() - t0) / 60:.1f} min")
dates = pd.DatetimeIndex(da["time"].values).normalize()
assert all(CLIM[v].shape[0] == dates.size for v in CLIM), "the variables do not have the same days"
TMAX, TMIN, PR = CLIM["tasmax"], CLIM["tasmin"], CLIM["pr"]
del CLIM
print(f"{dates[0].date()} -> {dates[-1].date()}, {dates.size} days")

## 2. Indicators per crop

Harvest years and seasons as `compute_bio_variables` of 1.1.d: a 2-year day axis (year hy − 1, then hy); a season with
maturity DOY > 365 starts in hy − 1, the others lie in hy; a season whose first year is missing is skipped.

In [ ]:
def ylen(y):
    return 366 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 365


def longest_run(mask):
    # longest run of consecutive True along the day axis, per cell
    run = np.zeros(mask.shape[1], np.int32)
    best = np.zeros(mask.shape[1], np.int32)
    for row in mask:
        run = np.where(row, run + 1, 0)
        np.maximum(best, run, out=best)
    return best


def indicators(sow, ant, mat, tcrit, tbase, winter):
    # sow, ant, mat: DOY per land cell (calendar of the ERA5 BIO file; maturity > 365 = season crossing New Year);
    # winter: cells of winter wheat (frost counted around anthesis, not from emergence)
    yr, doy = dates.year.values, dates.dayofyear.values
    years = set(yr)
    valid = np.isfinite(sow) & np.isfinite(ant) & np.isfinite(mat)
    wrap = valid & (mat > 365)
    s0, a0, m0 = (np.nan_to_num(x) for x in (sow, ant, mat))
    keys = ["HEAT_Y", "FROST_Y", "DRY_Y", "HEAT_D", "FROST_D", "DRY_D", "GDD"]
    acc, cnt = {k: np.zeros(NC) for k in keys}, np.zeros(NC)
    for hy in range(int(ERA5_PERIOD[0]) + 1, int(ERA5_PERIOD[1]) + 1):
        if hy not in years:
            continue
        ok = valid & (((hy - 1) in years) | ~wrap)
        idx = np.where((yr == hy) | (yr == hy - 1))[0]
        T = (doy[idx] + np.where(yr[idx] == hy, ylen(hy - 1), 0))[:, None]   # day on the 2-year axis
        shift = np.where(wrap, 0, ylen(hy - 1))[None]
        s, a, m = s0[None] + shift, a0[None] + shift, m0[None] + shift
        fw = np.where(winter[None], (T >= a + FLOWER_WINDOW[0]) & (T < a + FLOWER_WINDOW[1]),
                      (T >= s + EMERGENCE) & (T < a))                 # frost window
        p2 = (T >= a) & (T <= m)
        fl = (T >= a + FLOWER_WINDOW[0]) & (T < a + FLOWER_WINDOW[1])
        season = (T >= s) & (T <= m)
        tx, tn, pr = TMAX[idx], TMIN[idx], PR[idx]
        hot, frost = fl & (tx > tcrit), fw & (tn < FROST_T)
        spell = longest_run(p2 & (pr < DRY_PR))
        n_fl, n_fw, n_p2 = fl.sum(0), fw.sum(0), p2.sum(0)
        frac = lambda n, d: np.where(d > 0, n / np.maximum(d, 1), 0.0)
        vals = {"HEAT_Y": hot.any(0), "FROST_Y": frost.any(0), "DRY_Y": spell >= DRY_SPELL,
                "HEAT_D": frac(hot.sum(0), n_fl), "FROST_D": frac(frost.sum(0), n_fw), "DRY_D": frac(spell, n_p2),
                "GDD": (np.maximum(np.minimum((tx + tn) / 2, GDD_CAP) - tbase, 0.0) * season).sum(0)}
        for k in keys:
            acc[k] += np.where(ok, vals[k], 0.0)
        cnt += ok
    return {k: np.where(cnt > 0, acc[k] / np.maximum(cnt, 1), np.nan) for k in keys}


def to_grid(x):
    g = np.full(LAND.shape, np.nan, np.float32)
    g[I, J] = x
    return g


IND = {}
for crop in CROPS:
    t1 = time.time()
    with xr.open_dataset(ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc") as ds:
        sow, ant, mat = (ds[v].values[I, J].astype(float) for v in ("sowing_doy", "phase2_start_doy", "maturity_doy"))
    b = base_of(crop)
    lat_c = GRID["lat"][I]
    autumn = np.where(lat_c >= 0, (sow >= WINTER_SOWING["north"][0]) | (sow <= WINTER_SOWING["north"][1]),
                      (sow >= WINTER_SOWING["south"][0]) & (sow <= WINTER_SOWING["south"][1]))
    winter = np.full(NC, crop == "Wheat-winter_rf") | ((crop == "Wheat_rf") & autumn)
    ind = indicators(sow, ant, mat, HEAT_TCRIT[b], GDD_BASE[b], winter)
    IND[crop] = {k: to_grid(v) for k, v in ind.items()}
    xr.Dataset({k: (("lat", "lon"), v) for k, v in IND[crop].items()}, coords={"lat": GRID["lat"], "lon": GRID["lon"]},
               attrs={"crop": crop, "heat_tcrit": HEAT_TCRIT[b], "gdd_base": GDD_BASE[b], "flower_window": str(FLOWER_WINDOW),
                      "dry": f"pr < {DRY_PR} mm/day, spell >= {DRY_SPELL} days", "climate": f"{CLIMATE} {ERA5_PERIOD}"}
               ).to_netcdf(OUT / f"indicators_{TAG}_{crop}.nc")
    print(f"{crop}: indicators done, {int(winter.sum())} cells with the winter-wheat frost window ({(time.time() - t1) / 60:.1f} min)", flush=True)

## 3. Cross-validation of the three sets (folds and grid of 3.g)

In [ ]:
def table_hash(X, y):
    return hashlib.md5(np.ascontiguousarray(X, dtype=np.float64).tobytes() + np.asarray(y, np.int8).tobytes()).hexdigest()


def class_weight(y):
    return {1: 1.0, 0: (y == 1).sum() / (y == 0).sum()}


def fit_one_pass(X, y, cost, gamma, probability=False):
    return SVC(kernel="rbf", C=cost, gamma=gamma, class_weight=class_weight(y), probability=probability,
               random_state=SEED).fit(X, y)


def scores(y, score, pred):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    return {"AUC": roc_auc_score(y, score), "SENS": sens, "SPEC": spec}


def one_fold(X, y, tr, te, cost, gamma):
    s = StandardScaler().fit(X[tr])
    m = fit_one_pass(s.transform(X[tr]), y[tr], cost, gamma)
    Xte = s.transform(X[te])
    return scores(y[te], m.decision_function(Xte), m.predict(Xte))


def cv_grid(X, y):
    folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))
    tasks = [(c, g, k) for c in COST_GRID for g in GAMMA_GRID for k in range(N_FOLDS)]
    res = Parallel(n_jobs=N_JOBS)(delayed(one_fold)(X, y, *folds[k], c, g) for c, g, k in tasks)
    per_fold = pd.DataFrame([{"cost": c, "gamma": g, "fold": k, **r} for (c, g, k), r in zip(tasks, res)])
    return per_fold.groupby(["cost", "gamma"])[["AUC", "SENS", "SPEC"]].mean().reset_index()


def table(crop, extra):
    # training table of 1.1.d with the 10 BIO variables of the base models, + the indicators at the cell of each row
    b = joblib.load(MODELS / f"svm_model_{TAG}_{crop}.joblib")
    df = pd.read_csv(ERA5_DIR / f"training_dataset_{TAG}_{crop}.csv")
    ii = np.searchsorted(GRID["lat"], df.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], df.lon.values - 1e-6)
    for k in extra:
        df[k] = IND[crop][k][ii, jj]
    cols = list(b["bio_cols"]) + list(extra)
    assert df[cols].notna().all().all(), f"{crop}: missing values in {cols}"
    return b, df, cols, df[cols].values.astype(float), df["label"].values.astype(int), (ii, jj)


FITS, rows = {}, []
for crop in CROPS:
    for name, extra in [("base", [])] + list(SETS.items()):
        t1 = time.time()
        b, df, cols, X, y, cells = table(crop, extra)
        if name == "base":
            assert b.get("table_hash") == table_hash(X, y), f"{crop}: the 3.g model was not trained on this table"
            cv, bundle = b["cv"], b
            pm = MODELS / f"folds_{TAG}_{crop}.joblib"         # fold models of 4.g (cross-fitting), read if they match
            F0 = joblib.load(pm) if pm.exists() else {}
            fpath = pm if (F0.get("hash") == table_hash(X, y) and F0.get("cost") == b["best_cost"]
                           and F0.get("gamma") == b["best_gamma"]) else OUT / f"folds_base_{crop}.joblib"
        else:
            cv = cv_grid(X, y)
            cv.to_csv(OUT / f"cv_{name}_{crop}.csv", index=False)
            best = cv.loc[cv["AUC"].idxmax()]
            scaler = StandardScaler().fit(X)
            model = fit_one_pass(scaler.transform(X), y, best["cost"], best["gamma"], probability=True)
            bundle = {"model": model, "scaler": scaler, "bio_cols": cols, "best_cost": float(best["cost"]),
                      "best_gamma": float(best["gamma"]), "n_class1": int(y.sum()), "n_class0": int((y == 0).sum()), "cv": cv}
            joblib.dump(bundle, OUT / f"svm_model_{name}_{crop}.joblib")
            fpath = OUT / f"folds_{name}_{crop}.joblib"
        FITS[(crop, name)] = (bundle, cols, X, y, cells, cu.crossfit_models(fpath, X, y, bundle, N_FOLDS, SEED, N_JOBS))
        best = cv.loc[cv["AUC"].idxmax()]
        rows.append({"crop": crop, "set": name, "predictors": len(cols), "cost": best["cost"], "gamma": best["gamma"],
                     "CV AUC": best["AUC"], "CV SENS": best["SENS"], "CV SPEC": best["SPEC"],
                     "CV TSS": best["SENS"] + best["SPEC"] - 1})
        print(f"{crop:16s} {name:5s} AUC {best['AUC']:.3f} TSS {best['SENS'] + best['SPEC'] - 1:.3f} "
              f"({(time.time() - t1) / 60:.1f} min)", flush=True)
CV = pd.DataFrame(rows).set_index(["crop", "set"])

## 4. Suitability with each set (cross-fitted on the training cells, ECOCROP TMIN of 4.g, no novelty)

In [ ]:
rows, SUIT = [], {}
for crop in CROPS:
    with xr.open_dataset(ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc") as ds:
        tgrow = cu.growing_temperature(ds, cu.ECOCROP_TMIN[crop][1])
        area = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
        for name in ["base", *SETS]:
            bundle, cols, X, y, (ii, jj), Fcf = FITS[(crop, name)]
            n1, n0 = bundle["n_class1"], bundle["n_class0"]
            A = np.stack([ds[c].values if c.startswith("BIO") else IND[crop][c] for c in cols], axis=-1)
            ok = LAND & np.isfinite(A).all(axis=-1)
            p = np.full(LAND.shape, np.nan)
            p[ok] = bundle["model"].predict_proba(bundle["scaler"].transform(A[ok]))[:, 1]
            p[ii, jj] = cu.crossfit_proba(Fcf, X)                         # training cells: fold model that did not see them
            p = cu.balanced_proba(p, n1, n0)
            svm_ok = LAND & (np.nan_to_num(p) >= cu.CLIMATE_P)
            clim_ok = LAND & cu.climate_suitable(p, tgrow, crop, DT)
            SUIT[(crop, name)] = clim_ok
            pres, absn = y == 1, y == 0
            ct = clim_ok[ii, jj]
            sens, spec = ct[pres].mean(), 1 - ct[absn].mean()
            rows.append({"crop": crop, "set": name,
                         "MIRCA area suitable (%)": 100 * area[clim_ok].sum() / area.sum(),
                         "presence cells suitable (%)": 100 * sens, "absences called suitable (%)": 100 * (1 - spec),
                         "TSS training cells (cross-fitted, TMIN)": sens + spec - 1,
                         "climatically suitable land (Mha)": mha(clim_ok),
                         "N of 50 outside cropland suitable (Mha)": mha(clim_ok & N50_OUT),
                         "SVM-suitable land removed by TMIN (Mha)": mha(svm_ok & ~clim_ok)})
RES = CV.join(pd.DataFrame(rows).set_index(["crop", "set"]))
RES.to_csv(OUT / f"daily_indices_test_{SET}.csv")
with pd.option_context("display.max_rows", 200, "display.max_columns", 30, "display.width", 250):
    display(RES.round(3))
    print("difference to base")
    d = RES.drop(columns=["predictors", "cost", "gamma"])
    display((d - d.xs("base", level="set").reindex(d.index, level="crop")).drop("base", level="set").round(3))

## 5. Univariate AUC of each indicator (presences vs absences of the training table)

In [ ]:
rows = []
for crop in CROPS:
    _, _, _, y, (ii, jj), _ = FITS[(crop, "years")]
    for k in ["HEAT_Y", "FROST_Y", "DRY_Y", "HEAT_D", "FROST_D", "DRY_D", "GDD"]:
        x = IND[crop][k][ii, jj]
        rows.append({"crop": crop, "indicator": k, "AUC": roc_auc_score(y, x),
                     "mean presences": np.nanmean(x[y == 1]), "mean absences": np.nanmean(x[y == 0])})
UNI = pd.DataFrame(rows).set_index(["crop", "indicator"])
UNI.to_csv(OUT / f"daily_indices_univariate_{SET}.csv")
with pd.option_context("display.max_rows", 200, "display.width", 200):
    display(UNI["AUC"].unstack("indicator").round(3))

## 6. Maps, maize: the indicators (frequency of years) and the suitability, base vs years

In [ ]:
crop = "Maize_rf" if "Maize_rf" in CROPS else CROPS[0]
fig, axes = plt.subplots(2, 2, figsize=(22, 10))
for ax, k in zip(axes.ravel(), ["HEAT_Y", "FROST_Y", "DRY_Y", "GDD"]):
    im = ax.pcolormesh(GRID["lon"], GRID["lat"], IND[crop][k], cmap="viridis" if k == "GDD" else "magma_r",
                       vmin=0, vmax=None if k == "GDD" else 1, shading="nearest")
    ax.set_ylim(-60, 85); ax.set_title(f"{crop}: {k}", fontsize=11); plt.colorbar(im, ax=ax, shrink=0.8)
fig.savefig(OUT / f"indicators_{crop}.png", dpi=90, bbox_inches="tight"); plt.show()

cols4 = (["#e6e6e6", "#1a9641", "#d7191c", "#2c7bb6"], ["neither", "both", "base only", "years only"])
b0, b1 = SUIT[(crop, "base")], SUIT[(crop, "years")]
c = np.where(b0 & b1, 1.0, np.where(b0, 2.0, np.where(b1, 3.0, 0.0)))
fig, ax = plt.subplots(figsize=(13, 6))
ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(LAND, c, np.nan), vmin=-0.5, vmax=3.5, cmap=ListedColormap(cols4[0]), shading="nearest")
ax.set_ylim(-60, 85); ax.set_title(f"{crop}: climatically suitable, base vs + daily indicators (frequency of years)", fontsize=11)
ax.legend(handles=[Patch(color=a, label=l) for a, l in zip(*cols4)], loc="lower left", fontsize=9, framealpha=0.9)
fig.savefig(OUT / f"suitability_base_vs_years_{crop}.png", dpi=90, bbox_inches="tight"); plt.show()